In [69]:
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments

from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

from datasets import Dataset

import re
import unicodedata

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [11]:
def clean_text(text):
    # Normalize unicode
    text = unicodedata.normalize("NFKD", text)
    
    # Remove HTML tags like <br/>
    text = re.sub(r'<.*?>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www.\S+', '', text)
    
    # Remove non-printable characters
    text = ''.join(ch for ch in text if ch.isprintable())
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [9]:
# load_dataset??

In [13]:
df_train = pd.read_csv("./data/Covid/Corona_NLP_train.csv",encoding='latin-1')
df_test = pd.read_csv("./data/Covid/Corona_NLP_test.csv",encoding='latin-1')

In [15]:
df_train.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [17]:
df_train["Sentiment"].value_counts()

Sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

In [19]:
df_train = df_train[["OriginalTweet","Sentiment"]]
df_test = df_test[["OriginalTweet","Sentiment"]]

In [21]:
df_train.isnull().sum()

OriginalTweet    0
Sentiment        0
dtype: int64

In [25]:
df_train.duplicated().sum()

0

In [27]:
df_train["text"] = df_train["OriginalTweet"].apply(clean_text)

In [29]:
df_train.head()

,OriginalTweet,Sentiment,text
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,@MeNyrbie @Phil_Gahan @Chrisitv and and
1,advice Talk to your neighbours family to excha...,Positive,advice Talk to your neighbours family to excha...
2,Coronavirus Australia: Woolworths to give elde...,Positive,Coronavirus Australia: Woolworths to give elde...
3,My food stock is not the only one which is emp...,Positive,My food stock is not the only one which is emp...
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative,"Me, ready to go at supermarket during the #COV..."


In [37]:
df_train["OriginalTweet"][4]

"Me, ready to go at supermarket during the #COVID19 outbreak.\r\r\n\r\r\nNot because I'm paranoid, but because my food stock is litteraly empty. The #coronavirus is a serious thing, but please, don't panic. It causes shortage...\r\r\n\r\r\n#CoronavirusFrance #restezchezvous #StayAtHome #confinement https://t.co/usmuaLq72n"

In [39]:
df_train["text"][4]

"Me, ready to go at supermarket during the #COVID19 outbreak.Not because I'm paranoid, but because my food stock is litteraly empty. The #coronavirus is a serious thing, but please, don't panic. It causes shortage...#CoronavirusFrance #restezchezvous #StayAtHome #confinement"

In [43]:
le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["Sentiment"])

In [47]:
df_train["Sentiment"].value_counts().to_dict()

{'Positive': 11422,
 'Negative': 9917,
 'Neutral': 7713,
 'Extremely Positive': 6624,
 'Extremely Negative': 5481}

In [57]:
df_train["label"].value_counts().to_dict()

{4: 11422, 2: 9917, 3: 7713, 1: 6624, 0: 5481}

In [49]:
output_map = {'Positive': 0,
 'Negative': 1,
 'Neutral': 2,
 'Extremely Positive': 3,
 'Extremely Negative': 4}

In [53]:
df_train["label_1"] = df_train["Sentiment"].apply(lambda x :output_map[x])

In [55]:
df_train.head()

,OriginalTweet,Sentiment,text,label,label_1
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,@MeNyrbie @Phil_Gahan @Chrisitv and and,3,2
1,advice Talk to your neighbours family to excha...,Positive,advice Talk to your neighbours family to excha...,4,0
2,Coronavirus Australia: Woolworths to give elde...,Positive,Coronavirus Australia: Woolworths to give elde...,4,0
3,My food stock is not the only one which is emp...,Positive,My food stock is not the only one which is emp...,4,0
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative,"Me, ready to go at supermarket during the #COV...",0,4


In [59]:
df_train = df_train[["text","label"]]

In [67]:
train_df,val_df = train_test_split(df_train,test_size=0.1, random_state=42)

In [71]:
train_ds = Dataset.from_pandas(train_df[["text","label"]])
val_ds = Dataset.from_pandas(val_df[["text","label"]])

In [73]:
train_ds

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 37041
})

## Model

In [ ]:
#small-large
#cased-uncased

# cased 
# Phil_Gahan != phil_gahan

# Uncased 
# Phil_Gahan == phil_gahan

In [76]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [78]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

In [80]:
train_token = train_ds.map(tokenize, batched=True)
val_token = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/37041 [00:00<?, ? examples/s]

Map:   0%|          | 0/4116 [00:00<?, ? examples/s]

In [82]:
train_token

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 37041
})

In [84]:
df1 = train_token.to_pandas()

In [86]:
df1.head()

,text,label,__index_level_0__,input_ids,token_type_ids,attention_mask
0,"Gaisss! Please read this,and please limit your...",1,274,"[101, 11721, 14643, 2015, 999, 3531, 3191, 202...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,@mygovindia Today just after a week of lockdow...,2,26318,"[101, 1030, 2026, 3995, 6371, 9032, 2651, 2074...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,Tuskys partners with Amref to provide on groun...,3,5787,"[101, 10722, 5874, 2015, 5826, 2007, 2572, 289...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,@chrissyteigen are u doing ur own grocery shop...,2,20819,"[101, 1030, 3782, 6508, 2618, 29206, 2024, 105...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,UK Critical Care Nurse Cries at Empty SuperMar...,0,11031,"[101, 2866, 4187, 2729, 6821, 12842, 2012, 406...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [65]:
# DataCollatorWithPadding??

In [88]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [69]:
# data_collator

DataCollatorWithPadding(tokenizer=BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [92]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [98]:
training_args = TrainingArguments(
    "bert-covid-sentiments",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

In [100]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_token,
    eval_dataset=val_token,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/var/folders/gc/npgf58h95t16nm6m6d8q9pcr0000gn/T/ipykernel_2528/6968577.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
save_path = "covid-sentiment-bert-model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)